# LISS-IV Cloud Removal GAN - Automated Patch Dataset Generation Pipeline

This notebook processes raw LISS-IV satellite ZIP files, extracts Band 2 (Green), Band 3 (Red), and Band 4 (NIR), normalizes pixel intensities, extracts valid 256x256 clear patches, generates paired synthetic cloudy patches, and saves them as `.npy` array files for training Pix2Pix cGAN models.

In [ ]:
# Automatically install required packages if running in a fresh kernel
import sys
try:
    import rasterio
    import cv2
except ImportError:
    print("Installing missing dependencies (rasterio, opencv-python, numpy, matplotlib)...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rasterio", "opencv-python", "numpy", "matplotlib"])
    import rasterio
    import cv2

import os
import zipfile
import shutil
import glob
import gc
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Dataset directories
RAW_ZIPS_DIR = Path("Data/raw_Zips")
TEMP_EXTRACT_DIR = Path("temp_extract")
CLEAR_DIR = Path("Data/Clear_Patches")
CLOUDY_DIR = Path("Data/Cloudy_Patches")
PREVIEW_DIR = Path("Data/Preview_Images")

# Create output directories
CLEAR_DIR.mkdir(parents=True, exist_ok=True)
CLOUDY_DIR.mkdir(parents=True, exist_ok=True)
TEMP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

# Patch Extraction Settings
PATCH_SIZE = 256
STRIDE = 128  # 50% overlap for dataset generation
VALID_THRESHOLD = 0.90  # At least 90% non-zero valid pixels per patch
SCALE = 1  # Native 5.8m spatial resolution
MIN_STD_DEV = 0.01  # Minimum standard deviation to reject flat/featureless patches
MIN_MEAN_BRIGHTNESS = 0.02  # Minimum mean brightness to reject dark patches

print("Pipeline initialized.")
print(f"Zip directory: {RAW_ZIPS_DIR.resolve()}")
print(f"Clear patches destination: {CLEAR_DIR.resolve()}")
print(f"Cloudy patches destination: {CLOUDY_DIR.resolve()}")

In [ ]:
def enhance_band(path, scale=1):
    """
    Reads raster band, applies 2%-98% percentile linear stretch and normalizes to [0, 1].
    """
    with rasterio.open(path) as src:
        band = src.read(
            1,
            out_shape=(src.height // scale, src.width // scale),
            resampling=rasterio.enums.Resampling.bilinear
        ).astype(np.float32)
        
    valid_mask = band > 0
    if not np.any(valid_mask):
        return band
        
    p2 = np.percentile(band[valid_mask], 2)
    p98 = np.percentile(band[valid_mask], 98)
    
    if p98 > p2:
        band = (band - p2) / (p98 - p2)
        band = np.clip(band, 0, 1)
    else:
        band = np.zeros_like(band)
        
    band[~valid_mask] = 0
    return band

def generate_realistic_cloud_mask(h=256, w=256):
    """
    Generates multi-octave cloud opacity mask with smooth gradients and spatial variation.
    """
    # Octave 1: Large structural cloud shape (4x4 grid)
    n1 = np.random.uniform(0, 1, (4, 4)).astype(np.float32)
    m1 = cv2.resize(n1, (w, h), interpolation=cv2.INTER_CUBIC)
    
    # Octave 2: Medium frequency cloud features (8x8 grid)
    n2 = np.random.uniform(0, 1, (8, 8)).astype(np.float32)
    m2 = cv2.resize(n2, (w, h), interpolation=cv2.INTER_CUBIC)
    
    # Octave 3: Fine wispy details (16x16 grid)
    n3 = np.random.uniform(0, 1, (16, 16)).astype(np.float32)
    m3 = cv2.resize(n3, (w, h), interpolation=cv2.INTER_CUBIC)
    
    # Weighted multi-octave blend
    raw_mask = 0.65 * m1 + 0.25 * m2 + 0.10 * m3
    
    # Heavy Gaussian blur for natural soft cloud borders
    raw_mask = cv2.GaussianBlur(raw_mask, (51, 51), 0)
    
    # Normalize mask to [0, 1]
    r_min, r_max = raw_mask.min(), raw_mask.max()
    if r_max > r_min:
        raw_mask = (raw_mask - r_min) / (r_max - r_min)
        
    # Varied atmospheric cloud cover types
    cloud_type = np.random.choice(['wispy', 'bank', 'overcast'])
    if cloud_type == 'wispy':
        mask = np.power(raw_mask, 2.0) * np.random.uniform(0.3, 0.6)
    elif cloud_type == 'bank':
        angle = np.random.uniform(0, 2 * np.pi)
        x = np.linspace(-1, 1, w)
        y = np.linspace(-1, 1, h)
        xx, yy = np.meshgrid(x, y)
        gradient = np.cos(angle) * xx + np.sin(angle) * yy
        gradient = (gradient - gradient.min()) / (gradient.max() - gradient.min() + 1e-6)
        mask = raw_mask * gradient
        mask = np.power(mask, 1.4) * np.random.uniform(0.5, 0.85)
    else:
        mask = np.power(raw_mask, 1.2) * np.random.uniform(0.4, 0.85)
        
    return np.clip(mask, 0, 1)

def add_synthetic_clouds(clean_patch):
    """
    Applies realistic multi-octave cloud mask to clean ground patch.
    clean_patch shape: (3, H, W) [B2, B3, B4] in range [0, 1]
    """
    c, h, w = clean_patch.shape
    cloud_mask = generate_realistic_cloud_mask(h, w)
    alpha = np.expand_dims(cloud_mask, axis=0)
    
    # Natural bright white cloud reflection across bands [B2, B3, B4]
    cloud_color = np.array([
        np.random.uniform(0.85, 0.98),  # B2 Green
        np.random.uniform(0.85, 0.98),  # B3 Red
        np.random.uniform(0.85, 0.98)   # B4 NIR
    ], dtype=np.float32)[:, None, None]
    
    cloudy_patch = (1.0 - alpha) * clean_patch + alpha * cloud_color
    return np.clip(cloudy_patch, 0, 1)

print("Preprocessing and realistic cloud augmentation functions loaded successfully.")

In [ ]:
# Search for raw ZIP files
zip_files = list(RAW_ZIPS_DIR.glob("*.zip"))
if not zip_files:
    zip_files = list(Path("Data").rglob("*.zip"))

print(f"Found {len(zip_files)} raw ZIP scenes for processing.")

total_valid_pairs = 0
total_skipped_patches = 0

for zip_idx, zip_file in enumerate(zip_files, 1):
    print(f"\n[{zip_idx}/{len(zip_files)}] Processing Scene: {zip_file.name}")
    
    # Clear temp extraction directory
    if TEMP_EXTRACT_DIR.exists():
        shutil.rmtree(TEMP_EXTRACT_DIR)
    TEMP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Unzip scene
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(TEMP_EXTRACT_DIR)
        
    # Locate B2, B3, B4 TIFF files
    tif_files = list(TEMP_EXTRACT_DIR.rglob("*.tif")) + list(TEMP_EXTRACT_DIR.rglob("*.tiff"))
    band_paths = {}
    for tif in tif_files:
        fname = tif.name.upper()
        if "BAND2" in fname or "_B2" in fname or "BAND_2" in fname:
            band_paths["B2"] = tif
        elif "BAND3" in fname or "_B3" in fname or "BAND_3" in fname:
            band_paths["B3"] = tif
        elif "BAND4" in fname or "_B4" in fname or "BAND_4" in fname:
            band_paths["B4"] = tif
            
    if not ("B2" in band_paths and "B3" in band_paths and "B4" in band_paths):
        print(f"   [Warning] Missing bands in {zip_file.name}. Found: {list(band_paths.keys())}. Skipping.")
        continue
        
    # Enhance/read bands (Applies 2%-98% percentile normalization to [0, 1])
    b2_img = enhance_band(band_paths["B2"], scale=SCALE)
    b3_img = enhance_band(band_paths["B3"], scale=SCALE)
    b4_img = enhance_band(band_paths["B4"], scale=SCALE)
    
    h, w = b2_img.shape
    scene_valid_count = 0
    scene_skip_count = 0
    
    # Sliding window patch extraction
    for row in range(0, h - PATCH_SIZE + 1, STRIDE):
        for col in range(0, w - PATCH_SIZE + 1, STRIDE):
            patch_b2 = b2_img[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
            patch_b3 = b3_img[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
            patch_b4 = b4_img[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
            
            # Check valid non-zero mask across all 3 bands
            valid_pixels = (patch_b2 > 0) & (patch_b3 > 0) & (patch_b4 > 0)
            valid_frac = valid_pixels.mean()
            
            # Filter border patches AND hazy/uniform patches
            if valid_frac < VALID_THRESHOLD:
                scene_skip_count += 1
                continue
            
            clear_patch = np.stack([patch_b2, patch_b3, patch_b4], axis=0).astype(np.float32)
            
            # Statistical quality check (skip hazy patches)
            patch_std = clear_patch[:, valid_pixels].std()
            patch_mean = clear_patch[:, valid_pixels].mean()
            
            if patch_std < MIN_STD_DEV or patch_mean < MIN_MEAN_BRIGHTNESS:
                scene_skip_count += 1
                continue
            
            # Generate synthetic cloudy patch
            cloudy_patch = add_synthetic_clouds(clear_patch)
            
            # Sanity check (no NaNs, values in [0,1])
            if not (np.isfinite(clear_patch).all() and np.isfinite(cloudy_patch).all()):
                scene_skip_count += 1
                continue
            
            # Save array files
            file_id = f"patch_{total_valid_pairs:05d}.npy"
            np.save(CLEAR_DIR / file_id, clear_patch)
            np.save(CLOUDY_DIR / file_id, cloudy_patch)
            
            total_valid_pairs += 1
            scene_valid_count += 1
            
    print(f"   Saved {scene_valid_count} valid patch pairs. (Skipped {scene_skip_count} bad patches)")
    
    # Free memory between scenes
    del b2_img, b3_img, b4_img, patch_b2, patch_b3, patch_b4
    gc.collect()

# Clean up ONLY ONCE at the very end
if TEMP_EXTRACT_DIR.exists():
    shutil.rmtree(TEMP_EXTRACT_DIR)

print("\n" + "="*50)
print(f" Processing Complete! Total valid patch pairs generated: {total_valid_pairs}")
print("="*50)

## Dataset Sample & Normalization Verification
Visualize paired Clear (Ground Truth) vs Synthetic Cloudy Patches, verify normalized pixel ranges `[0.0, 1.0]`, and save viewable PNG images.

In [ ]:
clear_files = sorted(list(CLEAR_DIR.glob("*.npy")))
cloudy_files = sorted(list(CLOUDY_DIR.glob("*.npy")))

print(f"Total clear patches in directory: {len(clear_files)}")
print(f"Total cloudy patches in directory: {len(cloudy_files)}")

if clear_files:
    # Print Normalization Statistics for the first patch
    sample_patch = np.load(clear_files[0])
    print("\n--- Normalized Patch Pixel Range Inspection ---")
    print(f"Patch Shape: {sample_patch.shape} (Channels: 3 [B2, B3, B4], H: 256, W: 256)")
    for idx, band_name in enumerate(["B2 (Green)", "B3 (Red)", "B4 (NIR)"]):
        b_data = sample_patch[idx]
        print(f"  {band_name:12s} -> Min: {b_data.min():.4f}, Max: {b_data.max():.4f}, Mean: {b_data.mean():.4f}, Std: {b_data.std():.4f}")
    print("-" * 48)
    
    # Save viewable PNG previews to Data/Preview_Images/
    PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
    num_samples = min(6, len(clear_files))
    for i in range(num_samples):
        clear_p = np.load(clear_files[i])
        cloudy_p = np.load(cloudy_files[i])
        
        clear_fcc = np.stack([clear_p[2], clear_p[1], clear_p[0]], axis=-1)
        cloudy_fcc = np.stack([cloudy_p[2], cloudy_p[1], cloudy_p[0]], axis=-1)
        
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(clear_fcc)
        axes[0].set_title(f"Clear Ground Patch #{i:02d} (Normalized FCC)")
        axes[0].axis("off")
        
        axes[1].imshow(cloudy_fcc)
        axes[1].set_title(f"Synthetic Cloudy Patch #{i:02d}")
        axes[1].axis("off")
        
        plt.tight_layout()
        plt.savefig(PREVIEW_DIR / f"patch_pair_preview_{i:02d}.png", dpi=150)
        plt.close(fig)
        
    print(f"Saved {num_samples} viewable PNG images to: {PREVIEW_DIR.resolve()}")
    
    # Display in notebook
    fig, axes = plt.subplots(min(4, len(clear_files)), 2, figsize=(10, 4 * min(4, len(clear_files))))
    if min(4, len(clear_files)) == 1:
        axes = np.expand_dims(axes, axis=0)
        
    for i in range(min(4, len(clear_files))):
        clear_p = np.load(clear_files[i])
        cloudy_p = np.load(cloudy_files[i])
        
        clear_fcc = np.stack([clear_p[2], clear_p[1], clear_p[0]], axis=-1)
        cloudy_fcc = np.stack([cloudy_p[2], cloudy_p[1], cloudy_p[0]], axis=-1)
        
        axes[i, 0].imshow(clear_fcc)
        axes[i, 0].set_title(f"Clear Patch #{i:02d} (Normalized FCC)")
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(cloudy_fcc)
        axes[i, 1].set_title(f"Synthetic Cloudy Patch #{i:02d}")
        axes[i, 1].axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("No patches found to visualize. Run the processing cell above.")